In [12]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
import dagshub

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

# Paste the path you copied from the folder menu here
file_path = '/content/drive/MyDrive/Sentiment analysis /cleaned_data3.csv'

# Load the data into the 'df' variable
df = pd.read_csv(file_path)
df.dropna(inplace = True)
df["Sentiment"] = df["Sentiment"].astype("int8")

In [9]:
dagshub.init(repo_owner= 'vaibhav-patel01' , repo_name= 'YT_Comments_Analysis_chrome_extension' , mlflow= True)

mlflow.set_tracking_uri("https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow")
mlflow.set_experiment("finding_best_feature_engineering_algo")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=8e5e1415-fa7c-4b9c-bab1-689e6baa5b28&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=dc6789fac99de311644d9b3112613ff72ff11d3777421d3d351ba801bae4f31f




Accessing as vaibhav-patel01

Initialized MLflow to track repo "vaibhav-patel01/YT_Comments_Analysis_chrome_extension"

Repository vaibhav-patel01/YT_Comments_Analysis_chrome_extension initialized!

<Experiment: artifact_location='mlflow-artifacts:/cf5821c3c16a404c81ff005e5f1f60f0', creation_time=1783399519152, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783399519152, lifecycle_stage='active', name='finding_best_feature_engineering_algo', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [10]:

mlflow.set_experiment("selecting best algo")

2026/07/09 04:18:42 INFO mlflow.tracking.fluent: Experiment with name 'selecting best algo' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/8c2385d59dbe485781b9535aaf601075', creation_time=1783570722396, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783570722396, lifecycle_stage='active', name='selecting best algo', tags={}, trace_location=None, workspace='default'>

In [ ]:
# Suppress LightGBM warnings for cleaner Optuna output
warnings.filterwarnings("ignore", category=UserWarning)

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['Sentiment'] = df['Sentiment'].map({-1: 0, 0: 1, 1: 2})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['Sentiment'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 5000   # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['CommentText'])
y = df['Sentiment']

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Function to log results in MLflow
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, model={model_name}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

# Step 7: Run Optuna for LightGBM, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "LightGBM"
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-07-09 04:41:59,996] A new study created in memory with name: no-name-cc590d9e-66a3-46ca-a1ac-2a3c60758475
[I 2026-07-09 04:44:03,472] Trial 0 finished with value: 0.62023267017857 and parameters: {'n_estimators': 228, 'learning_rate': 0.0050819802652060115, 'max_depth': 8}. Best is trial 0 with value: 0.62023267017857.
[I 2026-07-09 04:45:28,288] Trial 1 finished with value: 0.5836262113302321 and parameters: {'n_estimators': 92, 'learning_rate': 0.00014763970163065822, 'max_depth': 9}. Best is trial 0 with value: 0.62023267017857.
[I 2026-07-09 04:46:31,588] Trial 2 finished with value: 0.5949280112465151 and parameters: {'n_estimators': 175, 'learning_rate': 0.007521687191525998, 'max_depth': 4}. Best is trial 0 with value: 0.62023267017857.
[I 2026-07-09 04:47:43,093] Trial 3 finished with value: 0.5750333683470624 and parameters: {'n_estimators': 112, 'learning_rate': 0.00014522803093146015, 'max_depth': 5}. Best is trial 0 with value: 0.62023267017857.
[I 2026-07-09 04:49:

🏃 View run LightGBM_TFIDF_Trigrams at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/3/runs/8c01a4294b09442ab4739c33ef503e9d
🧪 View experiment at: https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow/#/experiments/3


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['collections.OrderedDict', 'lightgbm.basic.Booster', 'lightgbm.sklearn.LGBMClassifier'].